In [1]:
import os
import time
import codecs
import json
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
from webdriver_manager.chrome import ChromeDriverManager
from urllib.parse import urljoin

In [2]:
# Tạo thư mục "crawl" nếu chưa có
output_dir = "crawl"
os.makedirs(output_dir, exist_ok=True)

In [3]:
# Khởi tạo trình duyệt Chrome
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Chạy nền
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
wait = WebDriverWait(driver, 10)


In [4]:
def extract_recipe_detail(url):
    try:
        driver.get(url)
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        soup = BeautifulSoup(driver.page_source, "html.parser")

        # Lấy tiêu đề món ăn
        title_tag = soup.find("span", class_="title !text-2xl lg:!text-5xl")
        title = title_tag.get_text(strip=True) if title_tag else ""

        # Hàm phụ lấy toàn bộ văn bản trong 1 section (bao gồm mọi nội dung lồng bên trong)
        def extract_full_text(section_id):
            section = soup.find("div", id=section_id)
            if section:
                return section.get_text(separator="\n", strip=True)
            return ""

        # Lấy nội dung từ các section theo id
        ingredients = extract_full_text("section-nguyenlieu")
        description = extract_full_text("section-soche")
        steps = extract_full_text("section-thuchien")
        usage = extract_full_text("section-howtouse")
        tips = extract_full_text("section-tips")

        return {
            "title": title,
            "description": description,
            "ingredients": ingredients,
            "steps": steps,
            "usage": usage,
            "tips": tips,
            "url": url
        }

    except Exception as e:
        print(f"❌ Lỗi khi trích xuất từ {url}: {e}")
        with open("crawl/failed_urls.txt", "a", encoding="utf-8") as log_file:
            log_file.write(url + "\n")
        return None


In [5]:
# Duyệt qua các trang
all_recipes = []
for page_num in range(1, 200):  # 1 đến 199
    print(f"🔄 Đang xử lý trang {page_num}...")
    page_url = f"https://monngonmoingay.com/tim-kiem-mon-ngon/page/{page_num}/"
    driver.get(page_url)
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
    soup = BeautifulSoup(driver.page_source, "html.parser")

    # Tìm tất cả nút "Xem chi tiết"
    detail_links = soup.find_all("a", class_="btn btn-primary btn-default btn-small flex-grow-0 uppercase")

    for a in detail_links:
        detail_url = a["href"]
        print(f"➡️ Truy cập: {detail_url}")
        recipe = extract_recipe_detail(detail_url)
        if recipe:
            all_recipes.append(recipe)

    time.sleep(1)  # Đợi tránh bị chặn

🔄 Đang xử lý trang 1...
➡️ Truy cập: https://monngonmoingay.com/bun-gao-xao-ga/
➡️ Truy cập: https://monngonmoingay.com/snack-salad/
➡️ Truy cập: https://monngonmoingay.com/banh-mi-ap-chao/
➡️ Truy cập: https://monngonmoingay.com/sup-gyoza/
➡️ Truy cập: https://monngonmoingay.com/vit-nuong-chao-2/
➡️ Truy cập: https://monngonmoingay.com/banh-qui-trai-cay/
➡️ Truy cập: https://monngonmoingay.com/chao-hau/
➡️ Truy cập: https://monngonmoingay.com/sandwiches-bo-xot-me-rang/
➡️ Truy cập: https://monngonmoingay.com/lau-nam/
➡️ Truy cập: https://monngonmoingay.com/mi-cay-chay-2/
➡️ Truy cập: https://monngonmoingay.com/lau-bo-khoai-cao/
➡️ Truy cập: https://monngonmoingay.com/lau-ca-thac-lac-kho-qua-2/
🔄 Đang xử lý trang 2...
➡️ Truy cập: https://monngonmoingay.com/mi-y-xot-cua-2/
➡️ Truy cập: https://monngonmoingay.com/gyoza-mi-pho-mai/
➡️ Truy cập: https://monngonmoingay.com/trung-hap/
➡️ Truy cập: https://monngonmoingay.com/cuon-trung-mayo/
➡️ Truy cập: https://monngonmoingay.com/canh-muop-

In [6]:
# Lưu kết quả ra file JSON
output_file = os.path.join(output_dir, "mon_ngon.json")
with codecs.open(output_file, "w", "utf-8-sig") as f:
    json.dump(all_recipes, f, ensure_ascii=False, indent=2)

print(f"\n✅ Đã lưu {len(all_recipes)} công thức vào {output_file}")
driver.quit()


✅ Đã lưu 2383 công thức vào crawl\mon_ngon.json


In [7]:
import csv

# Lưu thêm ra file CSV
csv_file = os.path.join(output_dir, "mon_ngon.csv")
with open(csv_file, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["title", "description", "ingredients", "steps", "usage", "tips", "url"])
    writer.writeheader()
    for recipe in all_recipes:
        writer.writerow(recipe)

print(f"📄 Đã lưu {len(all_recipes)} công thức vào {csv_file}")


📄 Đã lưu 2383 công thức vào crawl\mon_ngon.csv
